# Question 1 — function tests

Use this notebook to test each piece of `model.py` before training.

Kernel: select the **adl-cpu** interpreter.

Order: RNNCell → RNN → LSTMCell → LSTM → NextWordModel → dataset batch.

In [ ]:
import torch
print("torch", torch.__version__)
print("cuda ", torch.cuda.is_available())

torch 2.14.0+cpu
cuda  False


## Function 1 — RNNCell

One time step:

`h_t = tanh(W_x x_t + W_h h_prev + b)`

Shapes:
- `x_t`: (batch, input_size)
- `h_prev`: (batch, hidden_size)
- `h_t`: (batch, hidden_size)

In [ ]:
from model import RNNCell

batch_size = 2
input_size = 8
hidden_size = 16

cell = RNNCell(input_size=input_size, hidden_size=hidden_size, bias=True)

x_t = torch.randn(batch_size, input_size)
h_prev = torch.zeros(batch_size, hidden_size)
h_t = cell.forward(x_t=x_t, h_prev=h_prev)

print("x_t    ", tuple(x_t.shape))
print("h_prev ", tuple(h_prev.shape))
print("h_t    ", tuple(h_t.shape))
print("h_t range", float(h_t.min()), float(h_t.max()))

assert h_t.shape == (batch_size, hidden_size)
assert torch.all(h_t >= -1) and torch.all(h_t <= 1)
print("RNNCell test ok")

x_t     (2, 8)
h_prev  (2, 16)
h_t     (2, 16)
h_t range -0.7944241762161255 0.7983829975128174
RNNCell test ok


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_25616\3018347554.py:16: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  print("h_t range", float(h_t.min()), float(h_t.max()))


## Function 2 — RNN (loop over time)

Wait until Function 1 passes. Then we add `RNN` to `model.py` and test here.

Expected:
- `x`: (batch, time, input_size)
- `output`: (batch, time, hidden_size)

In [ ]:
import importlib
import model
importlib.reload(model)
from model import RNN

batch_size = 2
time_steps = 5
input_size = 8
hidden_size = 16

rnn = RNN(input_size=input_size, hidden_size=hidden_size, bias=True)
x = torch.randn(batch_size, time_steps, input_size)
output, h_last = rnn.forward(x=x, hidden=None)

print("x       ", tuple(x.shape))
print("output  ", tuple(output.shape))
print("h_last  ", tuple(h_last.shape))

assert output.shape == (batch_size, time_steps, hidden_size)
assert h_last.shape == (batch_size, hidden_size)
assert torch.allclose(output[:, -1, :], h_last)
print("RNN test ok")

x        (2, 5, 8)
output   (2, 5, 16)
h_last   (2, 16)
RNN test ok


## Function 3 — LSTMCell

Placeholder.

In [ ]:
import importlib
import model
importlib.reload(model)
from model import LSTMCell

batch_size = 2
input_size = 8
hidden_size = 16

cell = LSTMCell(input_size=input_size, hidden_size=hidden_size, bias=True)

x_t = torch.randn(batch_size, input_size)
h_prev = torch.zeros(batch_size, hidden_size)
c_prev = torch.zeros(batch_size, hidden_size)
h_t, c_t = cell.forward(x_t=x_t, h_prev=h_prev, c_prev=c_prev)

print("x_t ", tuple(x_t.shape))
print("h_t ", tuple(h_t.shape))
print("c_t ", tuple(c_t.shape))

assert h_t.shape == (batch_size, hidden_size)
assert c_t.shape == (batch_size, hidden_size)
assert torch.all(h_t >= -1) and torch.all(h_t <= 1)
print("LSTMCell test ok")

x_t  (2, 8)
h_t  (2, 16)
c_t  (2, 16)
LSTMCell test ok


## Function 4 — LSTM

Placeholder.

In [ ]:
import importlib
import model
importlib.reload(model)
from model import LSTM

batch_size = 2
time_steps = 5
input_size = 8
hidden_size = 16

lstm = LSTM(input_size=input_size, hidden_size=hidden_size, bias=True)
x = torch.randn(batch_size, time_steps, input_size)
output, (h_last, c_last) = lstm.forward(x=x, hidden=None)

print("x       ", tuple(x.shape))
print("output  ", tuple(output.shape))
print("h_last  ", tuple(h_last.shape))
print("c_last  ", tuple(c_last.shape))

assert output.shape == (batch_size, time_steps, hidden_size)
assert h_last.shape == (batch_size, hidden_size)
assert c_last.shape == (batch_size, hidden_size)
assert torch.allclose(output[:, -1, :], h_last)
print("LSTM test ok")

x        (2, 5, 8)
output   (2, 5, 16)
h_last   (2, 16)
c_last   (2, 16)
LSTM test ok


## Function 5 — NextWordModel + one dataset batch

Placeholder.

In [ ]:
import importlib
import model
importlib.reload(model)
from model import NextWordModel

batch_size = 2
time_steps = 6
vocab_size = 50
embed_size = 8
hidden_size = 16
pad_id = 0

model_lstm = NextWordModel(
    vocab_size=vocab_size,
    embed_size=embed_size,
    hidden_size=hidden_size,
    backbone="lstm",
    pad_id=pad_id,
    dropout=0.0,
)
model_lstm.eval()

input_ids = torch.randint(low=1, high=vocab_size, size=(batch_size, time_steps))
logits = model_lstm.forward(input_ids=input_ids)

print("input_ids", tuple(input_ids.shape))
print("logits   ", tuple(logits.shape))
assert logits.shape == (batch_size, time_steps, vocab_size)
print("NextWordModel LSTM test ok")

model_rnn = NextWordModel(
    vocab_size=vocab_size,
    embed_size=embed_size,
    hidden_size=hidden_size,
    backbone="rnn",
    pad_id=pad_id,
    dropout=0.0,
)
logits_rnn = model_rnn.forward(input_ids=input_ids)
assert logits_rnn.shape == (batch_size, time_steps, vocab_size)
print("NextWordModel RNN test ok")

input_ids (2, 6)
logits    (2, 6, 50)
NextWordModel LSTM test ok
NextWordModel RNN test ok


In [ ]:
from dataset import build_dataloaders

loaders, vocab, stats = build_dataloaders(batch_size=4, min_freq=2)
batch = next(iter(loaders["train"]))

net = NextWordModel(
    vocab_size=stats["vocab_size"],
    embed_size=64,
    hidden_size=128,
    backbone="lstm",
    pad_id=vocab.pad_id(),
    dropout=0.0,
)
net.eval()
logits = net.forward(input_ids=batch["input_ids"])
print("batch input", tuple(batch["input_ids"].shape))
print("batch labels", tuple(batch["labels"].shape))
print("logits      ", tuple(logits.shape))
assert logits.shape[0] == batch["input_ids"].shape[0]
assert logits.shape[1] == batch["input_ids"].shape[1]
assert logits.shape[2] == stats["vocab_size"]
print("dataset + model test ok")

batch input (4, 12)
batch labels (4, 12)
logits       (4, 12, 5960)
dataset + model test ok


# Function 6 — compute_loss

In [ ]:
import torch
from utils import compute_loss

batch_size = 2
time_steps = 4
vocab_size = 10
pad_id = 0

torch.manual_seed(0)
logits = torch.randn(batch_size, time_steps, vocab_size)
labels = torch.randint(low=1, high=vocab_size, size=(batch_size, time_steps))

# left-pad the first two positions of row 0
labels[0, 0] = pad_id
labels[0, 1] = pad_id

loss = compute_loss(logits=logits, labels=labels, pad_id=pad_id)
print("logits", tuple(logits.shape))
print("labels", labels.tolist())
print("loss  ", float(loss))

assert loss.ndim == 0
assert torch.isfinite(loss)
print("compute_loss test ok")

logits (2, 4, 10)
labels [[0, 0, 6, 8], [4, 3, 2, 8]]
loss   2.65897274017334
compute_loss test ok


# Function 7 — train_one_epoch 

In [ ]:
import torch
from dataset import build_dataloaders
from model import NextWordModel
from utils import train_one_epoch

loaders, vocab, stats = build_dataloaders(batch_size=16, min_freq=2)
device = torch.device("cpu")

net = NextWordModel(
    vocab_size=stats["vocab_size"],
    embed_size=64,
    hidden_size=128,
    backbone="lstm",
    pad_id=vocab.pad_id(),
    dropout=0.2,
).to(device)

optimizer = torch.optim.Adam(params=net.parameters(), lr=1e-3)

loss_before = train_one_epoch(
    model=net,
    loader=loaders["train"],
    optimizer=optimizer,
    pad_id=vocab.pad_id(),
    device=device,
    max_batches=5,
    grad_clip=1.0,
)
loss_after = train_one_epoch(
    model=net,
    loader=loaders["train"],
    optimizer=optimizer,
    pad_id=vocab.pad_id(),
    device=device,
    max_batches=5,
    grad_clip=1.0,
)

print("loss first 5 batches ", loss_before)
print("loss second 5 batches", loss_after)
assert loss_before > 0 and loss_after > 0
print("train_one_epoch test ok")

loss first 5 batches  8.682012939453125
loss second 5 batches 8.631362533569336
train_one_epoch test ok


# Function 8 — train_one_epoch 

In [ ]:
import torch
from dataset import build_dataloaders
from model import NextWordModel
from utils import evaluate, train_one_epoch

loaders, vocab, stats = build_dataloaders(batch_size=16, min_freq=2)
device = torch.device("cpu")

net = NextWordModel(
    vocab_size=stats["vocab_size"],
    embed_size=64,
    hidden_size=128,
    backbone="lstm",
    pad_id=vocab.pad_id(),
    dropout=0.0,
).to(device)

# a few train steps so metrics are not completely random
optimizer = torch.optim.Adam(params=net.parameters(), lr=1e-3)
train_one_epoch(
    model=net,
    loader=loaders["train"],
    optimizer=optimizer,
    pad_id=vocab.pad_id(),
    device=device,
    max_batches=10,
    grad_clip=1.0,
)

metrics = evaluate(
    model=net,
    loader=loaders["val"],
    pad_id=vocab.pad_id(),
    device=device,
    max_batches=10,
)

print(metrics)
assert metrics["loss"] > 0
assert metrics["ppl"] > 1
assert 0 <= metrics["top1"] <= 1
assert metrics["top5"] >= metrics["top1"]
print("evaluate test ok")

{'loss': 8.626386642456055, 'ppl': 5576.890625, 'top1': 0.0015625, 'top5': 0.0203125, 'n_tokens': 640, 'acc_at_k': {4: 0.0, 8: 0.0, 12: 0.0, 16: 0.0}}
evaluate test ok


# Function 9 — generate